In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.callbacks import EarlyStopping

import matplotlib.pyplot as plt


In [3]:
df = pd.read_csv("../Dataset/weekly_student_dataset.csv")

print(df.head())

print(df.shape)

print(df.columns.tolist())

   id_student code_module code_presentation  week  weekly_video_clicks  \
0        6516         AAA             2014J     1                  151   
1        6516         AAA             2014J     2                    9   
2        6516         AAA             2014J     3                   31   
3        6516         AAA             2014J     4                  241   
4        6516         AAA             2014J     5                   56   

   weekly_login_frequency  weekly_avg_activity_day  weekly_avg_quiz_score  \
0                       3               -21.280000                    0.0   
1                       2               -15.333333                    0.0   
2                       2                -5.333333                    0.0   
3                       5                -0.081081                    0.0   
4                       4                 5.333333                   60.0   

   weekly_assessments_completed  weekly_avg_submission_day  \
0                           0.

In [4]:
print(df.isnull().sum())

df = df.fillna(0)

print("Missing Values Removed")

id_student                      0
code_module                     0
code_presentation               0
week                            0
weekly_video_clicks             0
weekly_login_frequency          0
weekly_avg_activity_day         0
weekly_avg_quiz_score           0
weekly_assessments_completed    0
weekly_avg_submission_day       0
weekly_avg_assessment_weight    0
dropout                         0
dtype: int64
Missing Values Removed


In [5]:
label_encoder = LabelEncoder()

categorical_columns = [
    "code_module",
    "code_presentation"
]

for column in categorical_columns:

    df[column] = label_encoder.fit_transform(df[column])

print(df.head())

   id_student  code_module  code_presentation  week  weekly_video_clicks  \
0        6516            0                  3     1                  151   
1        6516            0                  3     2                    9   
2        6516            0                  3     3                   31   
3        6516            0                  3     4                  241   
4        6516            0                  3     5                   56   

   weekly_login_frequency  weekly_avg_activity_day  weekly_avg_quiz_score  \
0                       3               -21.280000                    0.0   
1                       2               -15.333333                    0.0   
2                       2                -5.333333                    0.0   
3                       5                -0.081081                    0.0   
4                       4                 5.333333                   60.0   

   weekly_assessments_completed  weekly_avg_submission_day  \
0                 

In [6]:
features = [

    "weekly_video_clicks",

    "weekly_login_frequency",

    "weekly_avg_activity_day",

    "weekly_avg_quiz_score",

    "weekly_assessments_completed",

    "weekly_avg_submission_day",

    "weekly_avg_assessment_weight",

    "code_module",

    "code_presentation"

]

target = "dropout"

In [7]:
scaler = StandardScaler()

df[features] = scaler.fit_transform(df[features])

print(df.head())

   id_student  code_module  code_presentation  week  weekly_video_clicks  \
0        6516     -1.85866           1.129025     1             0.921143   
1        6516     -1.85866           1.129025     2            -0.562918   
2        6516     -1.85866           1.129025     3            -0.332993   
3        6516     -1.85866           1.129025     4             1.861745   
4        6516     -1.85866           1.129025     5            -0.071715   

   weekly_login_frequency  weekly_avg_activity_day  weekly_avg_quiz_score  \
0                0.072551                -1.611406              -0.464081   
1               -0.485020                -1.533861              -0.464081   
2               -0.485020                -1.403461              -0.464081   
3                1.187695                -1.334972              -0.464081   
4                0.630123                -1.264368               1.476099   

   weekly_assessments_completed  weekly_avg_submission_day  \
0                 

In [8]:
sequence_length = 4

X = []
y = []

student_ids = []
prediction_weeks = []

In [9]:
students = df["id_student"].unique()

X = []
y = []

student_ids = []
prediction_weeks = []

for student in students:

    student_data = df[df["id_student"] == student]

    student_data = student_data.sort_values("week")

    values = student_data[features].values

    labels = student_data[target].values

    if len(values) < sequence_length + 1:
        continue

    for i in range(len(values) - sequence_length):

        X.append(values[i:i+sequence_length])

        y.append(labels[i + sequence_length])

        student_ids.append(student)

        prediction_weeks.append(
            student_data.iloc[i + sequence_length]["week"]
        )

X = np.array(X)

y = np.array(y)

print("X Shape :", X.shape)
print("y Shape :", y.shape)
print("Student IDs :", len(student_ids))
print("Prediction Weeks :", len(prediction_weeks))

X Shape : (530025, 4, 9)
y Shape : (530025,)
Student IDs : 530025
Prediction Weeks : 530025


In [10]:
(
    X_train,
    X_test,
    y_train,
    y_test,
    student_train,
    student_test,
    week_train,
    week_test
) = train_test_split(
    X,
    y,
    student_ids,
    prediction_weeks,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(X_train.shape)

print(X_test.shape)

(424020, 4, 9)
(106005, 4, 9)


In [11]:
class_weights = compute_class_weight(

    class_weight="balanced",

    classes=np.unique(y_train),

    y=y_train

)

class_weights = {

    0: class_weights[0],

    1: class_weights[1]

}

print(class_weights)

{0: np.float64(0.5462035491250851), 1: np.float64(5.910839745734359)}


In [12]:
model = Sequential()

model.add(

    LSTM(

        64,

        input_shape=(

            X_train.shape[1],

            X_train.shape[2]

        )

    )

)

model.add(

    Dropout(0.3)

)

model.add(

    Dense(

        32,

        activation="relu"

    )

)

model.add(

    Dropout(0.2)

)

model.add(

    Dense(

        1,

        activation="sigmoid"

    )

)

d:\nishanthini\Dropout prediction\project\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [13]:
model.compile(

    optimizer="adam",

    loss="binary_crossentropy",

    metrics=["accuracy"]

)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        18,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,057 (82.25 KB)

 Trainable params: 21,057 (82.25 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
early_stop = EarlyStopping(

    monitor="val_loss",

    patience=3,

    restore_best_weights=True

)

In [15]:
history = model.fit(

    X_train,

    y_train,

    validation_split=0.2,

    epochs=20,

    batch_size=32,

    class_weight=class_weights,

    callbacks=[early_stop],

    verbose=1

)

Epoch 1/20
10601/10601 ━━━━━━━━━━━━━━━━━━━━ 57s 5ms/step - accuracy: 0.6579 - loss: 0.5456 - val_accuracy: 0.7341 - val_loss: 0.4863
Epoch 2/20
10601/10601 ━━━━━━━━━━━━━━━━━━━━ 50s 5ms/step - accuracy: 0.6909 - loss: 0.5272 - val_accuracy: 0.7152 - val_loss: 0.5089
Epoch 3/20
10601/10601 ━━━━━━━━━━━━━━━━━━━━ 50s 5ms/step - accuracy: 0.7038 - loss: 0.5153 - val_accuracy: 0.7181 - val_loss: 0.5270
Epoch 4/20
10601/10601 ━━━━━━━━━━━━━━━━━━━━ 51s 5ms/step - accuracy: 0.7150 - loss: 0.5075 - val_accuracy: 0.7238 - val_loss: 0.5339


In [16]:
y_probability = model.predict(X_test).flatten()

print(y_probability[:10])

3313/3313 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step
[0.46935555 0.4835495  0.31645417 0.07338836 0.86059856 0.63873917
 0.03723635 0.16036591 0.3781256  0.02975056]


In [17]:
from sklearn.metrics import f1_score

thresholds = np.arange(

    0.10,

    0.91,

    0.05

)

best_threshold = 0.5

best_f1 = 0

for t in thresholds:

    y_pred = (

        y_probability >= t

    ).astype(int)

    score = f1_score(

        y_test,

        y_pred

    )

    print(

        f"Threshold={t:.2f}  F1={score:.4f}"

    )

    if score > best_f1:

        best_f1 = score

        best_threshold = t

print()

print("Best Threshold :", best_threshold)

print("Best F1 Score :", best_f1)

Threshold=0.10  F1=0.1982
Threshold=0.15  F1=0.2121
Threshold=0.20  F1=0.2250
Threshold=0.25  F1=0.2378
Threshold=0.30  F1=0.2506
Threshold=0.35  F1=0.2640
Threshold=0.40  F1=0.2780
Threshold=0.45  F1=0.2931
Threshold=0.50  F1=0.3115
Threshold=0.55  F1=0.3249
Threshold=0.60  F1=0.3325
Threshold=0.65  F1=0.3340
Threshold=0.70  F1=0.3227
Threshold=0.75  F1=0.2928
Threshold=0.80  F1=0.2396
Threshold=0.85  F1=0.1330
Threshold=0.90  F1=0.0309

Best Threshold : 0.6500000000000001
Best F1 Score : 0.3339864104381175


In [18]:
y_prediction = (

    y_probability >= best_threshold

).astype(int)

In [19]:
print(

    "Accuracy :",

    accuracy_score(

        y_test,

        y_prediction

    )

)

print(

    "Precision :",

    precision_score(

        y_test,

        y_prediction

    )

)

print(

    "Recall :",

    recall_score(

        y_test,

        y_prediction

    )

)

print(

    "F1 Score :",

    f1_score(

        y_test,

        y_prediction

    )

)

print(

    "ROC AUC :",

    roc_auc_score(

        y_test,

        y_probability

    )

)

print("\nConfusion Matrix")

print(

    confusion_matrix(

        y_test,

        y_prediction

    )

)

print("\nClassification Report")

print(

    classification_report(

        y_test,

        y_prediction

    )

)

Accuracy : 0.8492806943068724
Precision : 0.2666755425376115
Recall : 0.44674919147987063
F1 Score : 0.3339864104381175
ROC AUC : 0.804576832880359

Confusion Matrix
[[86022 11016]
 [ 4961  4006]]

Classification Report
              precision    recall  f1-score   support

           0       0.95      0.89      0.92     97038
           1       0.27      0.45      0.33      8967

    accuracy                           0.85    106005
   macro avg       0.61      0.67      0.62    106005
weighted avg       0.89      0.85      0.87    106005



In [20]:
future_risk = []

for prob in y_probability:

    if prob < 0.20:

        future_risk.append("Low")

    elif prob < 0.50:

        future_risk.append("Medium")

    else:

        future_risk.append("High")

In [21]:
intervention = []

for risk in future_risk:

    if risk == "Low":

        intervention.append(
            "Continue Current Learning Path"
        )

    elif risk == "Medium":

        intervention.append(
            "Weekly Mentor Follow-up"
        )

    else:

        intervention.append(
            "Immediate Faculty Intervention"
        )

In [22]:
future_prediction = pd.DataFrame({

    "Student ID": student_test,

    "Prediction Week": week_test,

    "Actual": y_test,

    "Predicted": y_prediction,

    "Risk Probability": y_probability,

    "Future Risk": future_risk,

    "Recommended Intervention": intervention

})

In [23]:
future_prediction = future_prediction.sort_values(
    by=["Student ID", "Prediction Week"]
)

future_prediction = future_prediction.groupby(
    "Student ID",
    as_index=False
).last()

print(future_prediction.shape)

(21180, 7)


In [24]:
future_prediction.to_csv(

    "../Dataset/future_risk_prediction.csv",

    index=False

)

print(

    "Future Risk Prediction Dataset Saved Successfully"

)

Future Risk Prediction Dataset Saved Successfully


In [25]:
print(

    future_prediction["Future Risk"].value_counts()

)

Future Risk
Low       13149
High       4020
Medium     4011
Name: count, dtype: int64


In [26]:
model.save(

    "../Models/future_risk_lstm_model.keras"

)

print(

    "LSTM Model Saved Successfully"

)

LSTM Model Saved Successfully


In [27]:
future_prediction.to_csv(
    "../Dataset/future_risk_prediction.csv",
    index=False
)

In [28]:
import joblib

joblib.dump(

    scaler,

    "../Models/lstm_scaler.pkl"

)

print(

    "Scaler Saved Successfully"

)

Scaler Saved Successfully


In [29]:
print("="*60)

print("Notebook 8 Completed Successfully")

print("="*60)

print("Model           : LSTM")

print("Sequence Length :", sequence_length)

print("Best Threshold  :", best_threshold)

print("ROC AUC         :", roc_auc_score(y_test, y_probability))

print("Prediction File : future_risk_prediction.csv")

print("Saved Model     : future_risk_lstm_model.keras")

print("="*60)

Notebook 8 Completed Successfully
Model           : LSTM
Sequence Length : 4
Best Threshold  : 0.6500000000000001
ROC AUC         : 0.804576832880359
Prediction File : future_risk_prediction.csv
Saved Model     : future_risk_lstm_model.keras
